In [ ]:
"""
FILE 5 — Evaluation and Visualization (fully compatible with File 4 output)
=============================================================================
Loads best_model_corrected.pt (saved by File 4), runs on test cases, produces:

  1. Metrics table   → results/rmse_table.csv
  2. Heatmaps        → results/heatmap_t***.png   (FEM | PINN | Error, 2×3 grid)
  3. Phase-front     → results/phase_front_2D.png  (smooth interpolated front)
  4. Melt fraction   → results/melt_fraction_evolution.png  (60°C only)
  5. R² scatter      → results/scatter_r2.png
  6. Animation       → results/pcm_melting_2d.gif

Run order: File 1 → File 2 → File 3 → File 4 → THIS FILE
"""

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import LinearSegmentedColormap
import pandas as pd
from pathlib import Path
import h5py

# ══════════════════════════════════════════════════════════════════════════════
# 0.  SIREN MODEL DEFINITION  (must match File 4 exactly)
# ══════════════════════════════════════════════════════════════════════════════

class SineLayer(nn.Module):
    def __init__(self, in_f, out_f, is_first=False, omega=30.0):
        super().__init__()
        self.omega  = omega
        self.linear = nn.Linear(in_f, out_f)
        with torch.no_grad():
            if is_first:
                self.linear.weight.uniform_(-1 / in_f, 1 / in_f)
            else:
                bound = np.sqrt(6 / in_f) / omega
                self.linear.weight.uniform_(-bound, bound)

    def forward(self, x):
        return torch.sin(self.omega * self.linear(x))


class PCM_SIREN(nn.Module):
    def __init__(self, in_dim=4, hidden=256, n_layers=6):
        super().__init__()
        layers = [SineLayer(in_dim, hidden, is_first=True)]
        for _ in range(n_layers - 1):
            layers.append(SineLayer(hidden, hidden))
        self.net     = nn.Sequential(*layers)
        self.head_T  = nn.Linear(hidden, 1)
        self.head_lf = nn.Sequential(nn.Linear(hidden, 1), nn.Sigmoid())
        self.head_u  = nn.Linear(hidden, 2)

    def forward(self, x):
        h   = self.net(x)
        T_n = self.head_T(h)
        lf  = self.head_lf(h)
        uv  = self.head_u(h)
        return torch.cat([T_n, lf, uv[:, 0:1], uv[:, 1:2]], dim=1)


# ══════════════════════════════════════════════════════════════════════════════
# 1.  SETUP
# ══════════════════════════════════════════════════════════════════════════════

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Path("results").mkdir(exist_ok=True)
print(f"Device: {DEVICE}")

# ── Load checkpoint saved by File 4 ──────────────────────────────────────────
CKPT_PATH = "best_model_corrected.pt"
ckpt      = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)

T_scale   = float(ckpt["T_scale"])
u_max     = float(ckpt["u_max"])
T_cold    = float(ckpt["T_cold"])
T_hot_min = float(ckpt["T_hot_min"])
T_hot_max = float(ckpt["T_hot_max"])

model = PCM_SIREN(hidden=256, n_layers=6).to(DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"Loaded model  epoch={ckpt['epoch']}  val_loss={ckpt['val_loss']:.5f}")

# ── Load HDF5 ground truth ────────────────────────────────────────────────────
DATA_PATH = r"C:\Users\raksh\pcm_2d_dataset.h5"
with h5py.File(DATA_PATH, "r") as f:
    T_gt   = f["T"][:]          # (N_cases, Nt, Ny, Nx)
    lf_gt  = f["lf"][:]
    ux_gt  = f["ux"][:]
    uy_gt  = f["uy"][:]
    t_arr  = f["t"][:]          # (Nt,)
    x_arr  = f["x"][:]          # (Nx,)
    y_arr  = f["y"][:]          # (Ny,)
    params = f["params"][:]     # (N_cases, n_params)
    W      = float(f.attrs["W"])
    H_dom  = float(f.attrs["H"])

Nx        = len(x_arr)
Ny        = len(y_arr)
Nt        = len(t_arr)
t_end     = float(t_arr[-1])
test_cases = [3, 7]             # 60 °C and 80 °C  — same as File 4

print(f"Grid: {Nx}×{Ny}, timesteps: {Nt}, t_end: {t_end:.0f}s")


# ══════════════════════════════════════════════════════════════════════════════
# 2.  PREDICTION HELPER
# ══════════════════════════════════════════════════════════════════════════════

def predict_field(case_idx: int, t_snap_idx: int):
    """
    Return T_pred, lf_pred, ux_pred, uy_pred — all in PHYSICAL units.
    Shapes: (Ny, Nx)
    """
    T_hot_c = params[case_idx, 0]
    t_n     = t_arr[t_snap_idx] / t_end
    thot_n  = (T_hot_c - T_hot_min) / (T_hot_max - T_hot_min + 1e-8)

    xx, yy  = np.meshgrid(x_arr / W, y_arr / H_dom)   # (Ny, Nx)
    x_flat  = xx.ravel().astype(np.float32)
    y_flat  = yy.ravel().astype(np.float32)
    t_flat  = np.full_like(x_flat, t_n)
    th_flat = np.full_like(x_flat, thot_n)

    inp = torch.tensor(
        np.stack([x_flat, y_flat, t_flat, th_flat], axis=1)
    ).to(DEVICE)

    with torch.no_grad():
        pred = model(inp).cpu().numpy()

    T_pred  = (pred[:, 0] * T_scale + T_cold).reshape(Ny, Nx)
    lf_pred =  pred[:, 1].reshape(Ny, Nx)
    ux_pred = (pred[:, 2] * u_max).reshape(Ny, Nx)
    uy_pred = (pred[:, 3] * u_max).reshape(Ny, Nx)

    return T_pred, lf_pred, ux_pred, uy_pred


# ══════════════════════════════════════════════════════════════════════════════
# 3.  METRIC HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def r2(true, pred):
    ss_res = np.sum((true - pred) ** 2)
    ss_tot = np.sum((true - np.mean(true)) ** 2)
    return 1.0 - ss_res / (ss_tot + 1e-10)

def rel_l2(true, pred):
    return np.sqrt(np.sum((true - pred) ** 2)) / (np.sqrt(np.sum(true ** 2)) + 1e-10)


# ══════════════════════════════════════════════════════════════════════════════
# 4.  COMPUTE METRICS ON TEST CASES
# ══════════════════════════════════════════════════════════════════════════════

print("\nComputing metrics on test cases …")

metrics = {k: [] for k in ["case", "T_hot_C", "RMSE_T", "RMSE_lf",
                             "RMSE_ux", "RMSE_uy", "R2_T", "R2_lf",
                             "relL2_T", "relL2_lf"]}

for c in test_cases:
    T_pred_c  = np.zeros((Nt, Ny, Nx))
    lf_pred_c = np.zeros((Nt, Ny, Nx))
    ux_pred_c = np.zeros((Nt, Ny, Nx))
    uy_pred_c = np.zeros((Nt, Ny, Nx))

    for ti in range(Nt):
        T_pred_c[ti], lf_pred_c[ti], ux_pred_c[ti], uy_pred_c[ti] = predict_field(c, ti)

    metrics["case"].append(int(c))
    metrics["T_hot_C"].append(float(params[c, 0] - 273.15))
    metrics["RMSE_T"].append(float(np.sqrt(np.mean((T_gt[c]  - T_pred_c) ** 2))))
    metrics["RMSE_lf"].append(float(np.sqrt(np.mean((lf_gt[c] - lf_pred_c) ** 2))))
    metrics["RMSE_ux"].append(float(np.sqrt(np.mean((ux_gt[c] - ux_pred_c) ** 2))))
    metrics["RMSE_uy"].append(float(np.sqrt(np.mean((uy_gt[c] - uy_pred_c) ** 2))))
    metrics["R2_T"].append(float(r2(T_gt[c].ravel(),  T_pred_c.ravel())))
    metrics["R2_lf"].append(float(r2(lf_gt[c].ravel(), lf_pred_c.ravel())))
    metrics["relL2_T"].append(float(rel_l2(T_gt[c].ravel(),  T_pred_c.ravel())))
    metrics["relL2_lf"].append(float(rel_l2(lf_gt[c].ravel(), lf_pred_c.ravel())))

df = pd.DataFrame(metrics)
df.to_csv("results/rmse_table.csv", index=False)
print(df.to_string(index=False))
print(f"\nMean RMSE_T  = {df['RMSE_T'].mean():.4f} K")
print(f"Mean RMSE_lf = {df['RMSE_lf'].mean():.4f}")
print(f"Mean R²_T    = {df['R2_T'].mean():.4f}")
print(f"Mean R²_lf   = {df['R2_lf'].mean():.4f}")


# ══════════════════════════════════════════════════════════════════════════════
# 5.  COLORMAPS
# ══════════════════════════════════════════════════════════════════════════════

cmap_T   = plt.cm.RdBu_r
cmap_lf  = LinearSegmentedColormap.from_list(
               "lf", ["#1a3c6b", "#87c4ff", "#ff9a3c", "#b30000"])
cmap_err = plt.cm.hot_r


# ══════════════════════════════════════════════════════════════════════════════
# 6.  HEATMAPS  —  FEM | PINN | |Error|   (2 rows × 3 cols)
# ══════════════════════════════════════════════════════════════════════════════

print("\nGenerating heatmap figures …")

snap_indices = [Nt // 4, Nt // 2, 3 * Nt // 4, Nt - 1]
c_plot       = test_cases[0]          # 60 °C test case
T_hot_plot   = params[c_plot, 0]
ext          = [0, W * 100, 0, H_dom * 100]   # axes in cm

for ti in snap_indices:
    T_p, lf_p, _, _ = predict_field(c_plot, ti)
    T_g  = T_gt[c_plot,  ti]
    lf_g = lf_gt[c_plot, ti]

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle(
        f"Case: T_hot={T_hot_plot - 273.15:.0f}°C  |  t={t_arr[ti]:.0f}s",
        fontsize=14, fontweight="bold")

    # ── Row 0 : Temperature ───────────────────────────────────────────────────
    vmin_T = T_cold - 273.15
    vmax_T = T_hot_plot - 273.15

    im00 = axes[0, 0].imshow(T_g  - 273.15, origin="lower", extent=ext,
                              cmap=cmap_T, vmin=vmin_T, vmax=vmax_T)
    axes[0, 0].set_title("FEM: Temperature (°C)")
    plt.colorbar(im00, ax=axes[0, 0])

    im01 = axes[0, 1].imshow(T_p  - 273.15, origin="lower", extent=ext,
                              cmap=cmap_T, vmin=vmin_T, vmax=vmax_T)
    axes[0, 1].set_title("PINN: Temperature (°C)")
    plt.colorbar(im01, ax=axes[0, 1])

    err_T = np.abs(T_g - T_p)
    im02  = axes[0, 2].imshow(err_T, origin="lower", extent=ext, cmap=cmap_err)
    axes[0, 2].set_title(
        f"|Error| T   RMSE={np.sqrt(np.mean(err_T**2)):.2f} K")
    plt.colorbar(im02, ax=axes[0, 2])

    # ── Row 1 : Liquid fraction ───────────────────────────────────────────────
    im10 = axes[1, 0].imshow(lf_g, origin="lower", extent=ext,
                              cmap=cmap_lf, vmin=0, vmax=1)
    axes[1, 0].set_title("FEM: Liquid Fraction")
    axes[1, 0].contour(lf_g, levels=[0.5], colors="white",
                       linewidths=1.5, extent=ext, origin="lower")
    plt.colorbar(im10, ax=axes[1, 0])

    im11 = axes[1, 1].imshow(lf_p, origin="lower", extent=ext,
                              cmap=cmap_lf, vmin=0, vmax=1)
    axes[1, 1].set_title("PINN: Liquid Fraction")
    axes[1, 1].contour(lf_p, levels=[0.5], colors="white",
                       linewidths=1.5, extent=ext, origin="lower")
    plt.colorbar(im11, ax=axes[1, 1])

    err_lf = np.abs(lf_g - lf_p)
    im12   = axes[1, 2].imshow(err_lf, origin="lower", extent=ext,
                                cmap=cmap_err, vmin=0, vmax=0.10)
    axes[1, 2].set_title(
        f"|Error| lf   RMSE={np.sqrt(np.mean(err_lf**2)):.4f}")
    plt.colorbar(im12, ax=axes[1, 2])

    for ax in axes.ravel():
        ax.set_xlabel("x (cm)")
        ax.set_ylabel("y (cm)")

    plt.tight_layout()
    fname = f"results/heatmap_t{ti:03d}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved {fname}  (t={t_arr[ti]:.0f}s)")


# ══════════════════════════════════════════════════════════════════════════════
# 7.  PHASE-FRONT POSITION vs y  —  smooth interpolated crossing
# ══════════════════════════════════════════════════════════════════════════════

def find_phase_front(lf_row: np.ndarray, x_coords: np.ndarray) -> float:
    """
    Return x (cm) where lf first crosses 0.5 from right (rightmost interface).
    Uses linear interpolation between adjacent cells — no zigzag artefacts.
    Returns np.nan if no crossing exists.
    """
    if lf_row.max() < 0.5 or lf_row.min() > 0.5:
        return np.nan
    crossings = np.where(np.diff(np.sign(lf_row - 0.5)))[0]
    if len(crossings) == 0:
        return np.nan
    i  = crossings[-1]                        # rightmost crossing index
    x0, x1 = x_coords[i], x_coords[i + 1]
    f0, f1 = lf_row[i],   lf_row[i + 1]
    x_cross = x0 + (0.5 - f0) * (x1 - x0) / (f1 - f0 + 1e-12)
    return float(x_cross * 100)               # metres → cm


print("\nGenerating phase-front plot …")

fig_pf, ax_pf = plt.subplots(figsize=(8, 6))
colors_t = plt.cm.plasma(np.linspace(0, 1, len(snap_indices)))
y_cm     = y_arr * 100

for idx, (ti, col) in enumerate(zip(snap_indices, colors_t)):
    _, lf_p, _, _ = predict_field(c_plot, ti)
    lf_g = lf_gt[c_plot, ti]

    pf_gt   = [find_phase_front(lf_g[j, :], x_arr) for j in range(Ny)]
    pf_pinn = [find_phase_front(lf_p[j, :], x_arr) for j in range(Ny)]

    lbl = f"t={t_arr[ti]:.0f}s"
    ax_pf.plot(pf_gt,   y_cm, color=col, ls="-",  lw=2.0,
               label=f"FEM  {lbl}")
    ax_pf.plot(pf_pinn, y_cm, color=col, ls="--", lw=2.0,
               label=f"PINN {lbl}")

# Colorbar for time
sm = plt.cm.ScalarMappable(
    cmap="plasma",
    norm=plt.Normalize(t_arr[snap_indices[0]], t_arr[snap_indices[-1]]))
plt.colorbar(sm, ax=ax_pf, label="Time (s)")

ax_pf.set_xlabel("Phase-front x position (cm)", fontsize=12)
ax_pf.set_ylabel("y (cm)", fontsize=12)
ax_pf.set_title(
    "Phase-front Position vs y\n(Curved front = natural convection confirmed)",
    fontsize=12)
ax_pf.legend(fontsize=7, ncol=2, loc="upper left")
ax_pf.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/phase_front_2D.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved results/phase_front_2D.png")


# ══════════════════════════════════════════════════════════════════════════════
# 8.  MELT FRACTION EVOLUTION  —  60 °C only (reliable test case)
# ══════════════════════════════════════════════════════════════════════════════

print("\nGenerating melt fraction evolution …")

fig_mf, ax_mf = plt.subplots(figsize=(8, 5))

c        = test_cases[0]          # case 3 = 60 °C
T_hot_c  = params[c, 0] - 273.15
mf_gt    = [lf_gt[c, ti].mean() for ti in range(Nt)]
mf_pinn  = [predict_field(c, ti)[1].mean() for ti in range(Nt)]

ax_mf.plot(t_arr, mf_gt,   ls="-",  lw=2.0, color="steelblue",
           label=f"FEM  T_h={T_hot_c:.0f}°C")
ax_mf.plot(t_arr, mf_pinn, ls="--", lw=2.0, color="darkorange",
           label=f"PINN T_h={T_hot_c:.0f}°C")

ax_mf.set_xlabel("Time (s)", fontsize=12)
ax_mf.set_ylabel("Mean Liquid Fraction", fontsize=12)
ax_mf.set_title("Melt Fraction Evolution: FEM vs PINN", fontsize=12)
ax_mf.legend(fontsize=10)
ax_mf.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/melt_fraction_evolution.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved results/melt_fraction_evolution.png")


# ══════════════════════════════════════════════════════════════════════════════
# 9.  R² SCATTER PLOT  —  Predicted vs True  (60 °C test case)
# ══════════════════════════════════════════════════════════════════════════════

print("\nGenerating R² scatter plot …")

c = test_cases[0]
T_pred_all, lf_pred_all = [], []
T_gt_all,   lf_gt_all   = [], []

for ti in range(0, Nt, 4):          # every 4th timestep → fast but representative
    T_p, lf_p, _, _ = predict_field(c, ti)
    T_pred_all.append(T_p.ravel())
    lf_pred_all.append(lf_p.ravel())
    T_gt_all.append(T_gt[c,  ti].ravel())
    lf_gt_all.append(lf_gt[c, ti].ravel())

T_pred_all  = np.concatenate(T_pred_all)
lf_pred_all = np.concatenate(lf_pred_all)
T_gt_all    = np.concatenate(T_gt_all)
lf_gt_all   = np.concatenate(lf_gt_all)

fig_sc, axes_sc = plt.subplots(1, 2, figsize=(12, 5))

# ── Temperature ───────────────────────────────────────────────────────────────
axes_sc[0].scatter(T_gt_all - 273.15, T_pred_all - 273.15,
                   alpha=0.05, s=2, color="steelblue", rasterized=True)
lims_T = [T_cold - 273.15, params[c, 0] - 273.15]
axes_sc[0].plot(lims_T, lims_T, "r--", lw=1.5, label="Perfect (y=x)")
r2_T = r2(T_gt_all, T_pred_all)
axes_sc[0].set_xlabel("FEM Temperature (°C)", fontsize=11)
axes_sc[0].set_ylabel("PINN Temperature (°C)", fontsize=11)
axes_sc[0].set_title(f"Temperature   R²={r2_T:.4f}", fontsize=12)
axes_sc[0].legend(fontsize=9)
axes_sc[0].grid(True, alpha=0.3)

# ── Liquid fraction ───────────────────────────────────────────────────────────
axes_sc[1].scatter(lf_gt_all, lf_pred_all,
                   alpha=0.05, s=2, color="darkorange", rasterized=True)
axes_sc[1].plot([0, 1], [0, 1], "r--", lw=1.5, label="Perfect (y=x)")
r2_lf = r2(lf_gt_all, lf_pred_all)
axes_sc[1].set_xlabel("FEM Liquid Fraction", fontsize=11)
axes_sc[1].set_ylabel("PINN Liquid Fraction", fontsize=11)
axes_sc[1].set_title(f"Liquid Fraction   R²={r2_lf:.4f}", fontsize=12)
axes_sc[1].legend(fontsize=9)
axes_sc[1].grid(True, alpha=0.3)

fig_sc.suptitle(
    "PINN vs FEM — Predicted vs True  (60°C test case)",
    fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("results/scatter_r2.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"  Saved results/scatter_r2.png   R²_T={r2_T:.4f}  R²_lf={r2_lf:.4f}")


# ══════════════════════════════════════════════════════════════════════════════
# 10.  ANIMATION  —  PINN prediction for 60 °C case
# ══════════════════════════════════════════════════════════════════════════════

print("\nGenerating animation (this may take a minute) …")

c_anim = test_cases[0]
ext    = [0, W * 100, 0, H_dom * 100]

frames_T, frames_lf = [], []
for ti in range(Nt):
    T_p, lf_p, _, _ = predict_field(c_anim, ti)
    frames_T.append(T_p  - 273.15)
    frames_lf.append(lf_p)

T_vmin = T_cold - 273.15
T_vmax = params[c_anim, 0] - 273.15

fig_anim, axes_anim = plt.subplots(1, 2, figsize=(10, 4))
fig_anim.suptitle(
    f"PINN: PCM Melting  T_hot={params[c_anim, 0] - 273.15:.0f}°C",
    fontsize=12)

im_T  = axes_anim[0].imshow(frames_T[0],  origin="lower", extent=ext,
                              cmap=cmap_T,  vmin=T_vmin, vmax=T_vmax,
                              animated=True)
im_lf = axes_anim[1].imshow(frames_lf[0], origin="lower", extent=ext,
                              cmap=cmap_lf, vmin=0, vmax=1,
                              animated=True)

axes_anim[0].set_title("Temperature (°C)")
axes_anim[0].set_xlabel("x (cm)")
axes_anim[0].set_ylabel("y (cm)")
axes_anim[1].set_title("Liquid Fraction")
axes_anim[1].set_xlabel("x (cm)")
axes_anim[1].set_ylabel("y (cm)")

plt.colorbar(im_T,  ax=axes_anim[0], fraction=0.03)
plt.colorbar(im_lf, ax=axes_anim[1], fraction=0.03)

time_label = fig_anim.text(0.5, 0.01, "t = 0 s", ha="center", fontsize=10)
plt.tight_layout()


def update(frame):
    im_T.set_data(frames_T[frame])
    im_lf.set_data(frames_lf[frame])
    time_label.set_text(f"t = {t_arr[frame]:.0f} s")
    return [im_T, im_lf, time_label]


ani = animation.FuncAnimation(
    fig_anim, update, frames=Nt, interval=150, blit=True)
ani.save("results/pcm_melting_2d.gif", writer="pillow", fps=6, dpi=100)
plt.close()
print("  Saved results/pcm_melting_2d.gif")


# ══════════════════════════════════════════════════════════════════════════════
# DONE
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("All results saved to  results/")
print("=" * 60)
print("  rmse_table.csv          ← quantitative metrics")
print("  heatmap_t***.png        ← FEM | PINN | Error heatmaps")
print("  phase_front_2D.png      ← smooth phase-front vs y")
print("  melt_fraction_evolution.png  ← melt fraction over time")
print("  scatter_r2.png          ← R² scatter (T and lf)")
print("  pcm_melting_2d.gif      ← animation")